# C++ and Lua Code Assistant — SmolLM2-360M Fine-Tune

**What this does:** Fine-tunes `HuggingFaceTB/SmolLM2-360M` (360M params) on C++ and Lua code using QLoRA.

**Hardware needed:** Free T4 GPU (12GB VRAM) — runs on Colab free tier.

**Output:** A LoRA adapter you can download (~30MB).

## 1. Install Dependencies

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

!pip install -qU \
    torch>=2.1.0 \
    transformers>=4.38.0 \
    accelerate>=0.27.0 \
    peft>=0.8.0 \
    bitsandbytes>=0.42.0 \
    trl>=0.8.0 \
    datasets>=2.16.0 \
    scipy>=1.11.0 \
    wandb \
    pyyaml

## 2. Clone from GitHub

Set `REPO_URL` to your GitHub repo URL below, then run the cell.

In [ ]:
REPO_URL = "https://github.com/zessdemon12-sudo/cpp-lua-code-assistant.git"

import os, sys, shutil
from pathlib import Path

!git clone {REPO_URL} /content/cpp-lua-code-assistant

PROJECT_DIR = Path("/content/cpp-lua-code-assistant")
os.chdir(str(PROJECT_DIR))
print(f"Project root: {PROJECT_DIR}")
!ls -la

## 3. Mount Google Drive (for saving results)

Optional — only if you want to save the trained model to Drive.

In [ ]:
MOUNT_DRIVE = True  # set to False if you don't want to mount

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = Path("/content/drive/MyDrive/cpp-lua-code-assistant")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Results will save to: {DRIVE_DIR}")
else:
    print("Drive not mounted — model will only be downloadable via the download cell")

## 4. Prepare Datasets

Downloads C++ and Lua code from The Stack v2, deduplicates, and generates instruction→code pairs.

In [ ]:
%%time
print("=== Collecting raw C++ and Lua code ===")
!python data/collect_data.py

print("\n=== Generating instruction→code pairs ===")
!python data/generate_instructions.py

## 5. Check GPU

Verify we're running on a GPU with enough VRAM.

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

if not torch.cuda.is_available():
    print("\n⚠️  No GPU detected! Go to Runtime > Change runtime type > T4 GPU")
    raise SystemExit("Need GPU to train.")

## 6. Log in to Weights & Biases (optional)

Get your API key from [wandb.ai/authorize](https://wandb.ai/authorize). Skip with Enter for no logging.

In [ ]:
wandb_key = getpass.getpass("W&B API key (or press Enter to skip): ")
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    !wandb login --relogin
    print("W&B logging enabled")
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("W&B disabled — training will still work")

## 7. Run Training

Fine-tunes SmolLM2-360M with QLoRA.
- ~2-3 hours on T4 GPU
- Uses ~8GB VRAM

In [ ]:
%%time
!python scripts/train.py
print("\n✅ Training complete!")

## 8. Evaluate the Model

Runs 20 prompts (10 C++, 10 Lua) and reports stats.

In [ ]:
!python scripts/evaluate.py

## 9. Test Inference

In [ ]:
print("="*60)
!python scripts/inference.py --lang cpp --prompt "Write a C++ function that implements a basic smart pointer"

print("\n" + "="*60 + "\n")

!python scripts/inference.py --lang lua --prompt "Write a Lua function that implements a simple event system"

## 10. Save & Download

Choose how to save your trained model:

In [ ]:
ADAPTER_PATH = PROJECT_DIR / "outputs" / "smollm2-360m-cpp-lua" / "final"

if not ADAPTER_PATH.exists():
    print("No trained adapter found. Run the training cell first.")
else:
    size_mb = sum(f.stat().st_size for f in ADAPTER_PATH.rglob("*")) / 1e6
    print(f"Adapter found: {ADAPTER_PATH}")
    print(f"Size: {size_mb:.1f} MB")

In [ ]:
# Option A: Save to Google Drive
if MOUNT_DRIVE and ADAPTER_PATH.exists():
    shutil.copytree(ADAPTER_PATH, DRIVE_DIR / "final", dirs_exist_ok=True)
    print(f"Saved to Drive: {DRIVE_DIR / 'final'}")
elif not MOUNT_DRIVE:
    print("Drive not mounted — use Option B (download)")
else:
    print("No trained adapter yet.")

In [ ]:
# Option B: Download as zip
if ADAPTER_PATH.exists():
    !cd "{PROJECT_DIR}" && zip -r /content/adapter.zip outputs/smollm2-360m-cpp-lua/final/
    from google.colab import files
    files.download('/content/adapter.zip')
    print("✅ Downloaded! Extract and use:")
    print(f"   python scripts/inference.py --adapter ./final")